# Plan de Tesis y Protocolo de Investigación (Informe Parcial — PI2)

**Título de tesis:** Optimización del Ruteo de Bandejas Eléctricas en Entornos OpenBIM mediante Voxelización Semántica, Búsqueda Heurística y Aprendizaje por Refuerzo Profundo  
**Programa:** Maestría en Inteligencia Artificial — Universidad Nacional de Ingeniería (UNI)  
**Tesista:** Ing. Juan José Villegas Calvay ([Juanvc07](https://github.com/Juanvc07))  
**Asesor:** María Fernanda Tejada Begazo  
**Docente del curso:** Dr. Glen Darío Rodríguez Rafael  
**Fecha:** Lima, noviembre de 2026

---

## 1. Resumen Ejecutivo y Arquitectura Metodológica

El presente cuaderno consolida el **Plan de Tesis y Protocolo Experimental** orientado a automatizar el trazado tridimensional de **bandejas eléctricas portacables** para alimentadores principales en edificaciones hospitalarias de alta complejidad (**Categoría III-1**), operando directamente sobre modelos interoperables **OpenBIM (IFC 4 / ISO 16739)**.

### Arquitectura Híbrida en Dos Fases
1. **Fase 1 (Completada — Caracterización Físico-Normativa Tabular):** Modelado de las restricciones electromecánicas sobre los 24 circuitos alimentadores reales (`data/schedules/cable_schedule.csv`, conductores libres de halógenos N2XOH 0.6/1 kV):
   * Control de multicolinealidad mediante Factor de Inflación de Varianza (**VIF**), descartando `seccion_mm2` (VIF = 1,914.96).
   * Ingeniería de atributos normativos: ratio de ocupación en bandeja (`R_fill_ref`, NEC Art. 392 ≤ 40%), carga mecánica logarítmica (`log_peso_total`) y margen de seguridad de curvatura (`MS_curv_ref`) frente a accesorios comerciales estándar de 300 mm.
   * Protocolo estricto de **cero fuga de información (*zero data leakage*)** con `MinMaxScaler` ajustado exclusivamente sobre entrenamiento dentro de un `Pipeline` en validación cruzada de 3 pliegues.
2. **Fase 2 (En Desarrollo — Motor Espacial OpenBIM + IA):**
   * **Ingesta OpenBIM (`IfcOpenShell`):** Extracción de mallas B-Rep de elementos estructurales (vigas, losas, columnas), arquitectónicos y MEP (ductos HVAC, tuberías).
   * **Voxelización Semántica 3D (50 mm) y Campos ESDF:** Cálculo de distancias euclidianas exactas hacia obstáculos para verificación en tiempo constante O(1) de colisiones y holgura superior libre de 300 mm (**NEMA VE 2**).
   * **Guía Macro-Heurística A* 3D Ponderada:** Generación de campo potencial denso para evitar recompensas dispersas en corredores hospitalarios largos.
   * **Agente PPO con Enmascaramiento Cinemático (*Action Masking*):** Restricción estricta de deflexiones a accesorios comerciales estándar (30°, 45°, 60° y 90°), respeto del radio mínimo de curvatura y bonificación por consolidación en troncales compartidas (*bundling*).

## 2. Matriz de Consistencia Metodológica (Versión Actualizada)

| Problema | Objetivo | Hipótesis | Variable | Indicador medible |
| :--- | :--- | :--- | :--- | :--- |
| **Problema General:**<br>¿En qué medida la integración de voxelización semántica, búsqueda heurística y aprendizaje por refuerzo profundo (PPO) sobre modelos OpenBIM (IFC) permite optimizar el trazado 3D de bandejas eléctricas de alimentadores en edificaciones hospitalarias de alta complejidad (Categoría III-1), eliminando interferencias espaciales y reduciendo los tiempos de diseño respecto al método manual y a la búsqueda heurística aislada? | **Objetivo General:**<br>Desarrollar e implementar un modelo híbrido de optimización basado en voxelización semántica, búsqueda heurística y aprendizaje por refuerzo profundo (PPO con enmascaramiento de acciones) sobre estándar OpenBIM (IFC 4 / ISO 16739) para automatizar el ruteo tridimensional de bandejas eléctricas de alimentadores en hospitales Categoría III-1, logrando cero colisiones geométricas, estricto cumplimiento normativo (CNE-Utilización, NEC Art. 392 y NEMA VE 2) y reducción del tiempo de trazado frente al diseño manual. | **Hipótesis General:**<br>La integración de un entorno espacial voxelizado semánticamente con campos ESDF, guiado por búsqueda heurística y optimizado mediante PPO con restricciones normativas explícitas, permite generar trazados 3D de bandejas eléctricas libres de colisiones (0 clashes), con 100% de cumplimiento de radio de curvatura, holgura superior NEMA VE 2 (≥ 300 mm) y llenado ≤ 40%, reduciendo en ≥ 60% el tiempo de diseño frente al trazado manual. | **Variable Independiente (X):**<br>Arquitectura híbrida de ruteo 3D (Voxelización Semántica IFC + Guía Heurística A* + Agente PPO con Action Masking).<br><br>**Variable Dependiente (Y):**<br>Desempeño técnico y constructabilidad del trazado de bandejas eléctricas. | • Conteo de colisiones duras (hard clashes = 0).<br>• Cumplimiento de holgura superior NEMA VE 2 (≥ 300 mm).<br>• Cumplimiento de radio mínimo de curvatura (R ≥ Rmín, 100%).<br>• Factor de llenado de bandeja (Rfill_ref ≤ 40%).<br>• Longitud total de bandeja (m) y tiempo de cómputo (s). |
| **Problema Específico 1:**<br>¿Cómo formular y validar un modelo de caracterización físico-normativa de los conductores y bandejas eléctricas que elimine la multicolinealidad entre parámetros dimensionales y cuantifique las variables críticas de ocupación y rigidez mecánica para alimentar la función de recompensa del agente? | **Objetivo Específico 1:**<br>Formular y validar mediante análisis de colinealidad (VIF) e ingeniería de atributos un modelo de caracterización físico-normativa de los circuitos alimentadores eléctricos, seleccionando con evidencia estadística las variables gobernantes de ocupación en bandeja (`R_fill_ref`), carga mecánica (`log_peso_total`) y margen de curvatura (`MS_curv_ref`) para parametrizar las restricciones y recompensas del entorno 3D. | **Hipótesis Específica 1:**<br>La depuración de variables colineales (VIF < 10) y la incorporación de atributos normativos derivados (`R_fill_ref`, `MS_curv_ref` y carga logarítmica) reducen el error de predicción del esfuerzo físico-constructivo respecto a las variables crudas del cuadro de cargas y concentran más del 80% de la importancia explicativa (Gain), proveyendo una señal de recompensa físicamente consistente. | **Variables Independientes (X1):**<br>• Ratio de ocupación (`R_fill_ref`)<br>• Margen de curvatura (`MS_curv_ref`)<br>• Carga lineal (`log_peso_total`)<br>• Diámetro exterior depurado por VIF<br><br>**Variable Dependiente (Y1):**<br>Índice de esfuerzo físico y ocupación normativa (`y_esfuerzo`). | • Factor de Inflación de Varianza (VIF < 10 en variables retenidas).<br>• Importancia relativa acumulada (Gain > 80% en variables creadas).<br>• Error Absoluto Medio (MAE) y RMSE en 3-Fold CV sin data leakage. |
| **Problema Específico 2:**<br>¿De qué manera la voxelización semántica y el cálculo de campos de distancia signada euclidiana (ESDF) a partir de geometrías OpenBIM (IFC) permiten representar los plenos técnicos hospitalarios saturados para detectar colisiones y holguras normativas (300 mm NEMA VE 2) de forma eficiente? | **Objetivo Específico 2:**<br>Construir un motor de ingestión y voxelización semántica 3D que extraiga elementos arquitectónicos, estructurales y MEP desde modelos IFC (`IfcOpenShell`) y calcule campos de distancia euclidiana (ESDF) para evaluar colisiones y holguras mínimas en tiempo real durante la exploración del agente. | **Hipótesis Específica 2:**<br>La discretización espacial mediante voxelización semántica multiclase (resolución ≤ 50 mm) acoplada a campos continuos ESDF permite consultar distancias a obstáculos en tiempo constante O(1) por paso, garantizando cero intersecciones geométricas y el respeto de la holgura superior libre de 300 mm (NEMA VE 2). | **Variables Independientes (X2):**<br>• Resolución de grilla voxelizada (Δv)<br>• Etiqueta semántica de entidad IFC<br><br>**Variable Dependiente (Y2):**<br>Detección de interferencias y gálibo libre en plenos técnicos. | • Tiempo de consulta espacial ESDF (ms, orden O(1)).<br>• Distancia libre vertical a losa/fondo de viga (≥ 300 mm).<br>• Conteo de intersecciones geométricas con ductos/tuberías MEP. |
| **Problema Específico 3:**<br>¿Cómo integrar una guía de búsqueda heurística con una política de aprendizaje por refuerzo profundo (PPO) dotada de enmascaramiento de acciones (action masking) para generar trayectorias de bandejas eléctricas compatibles con accesorios comerciales estándar (30°, 45°, 60° y 90°) y criterios de empaquetamiento (bundling)? | **Objetivo Específico 3:**<br>Diseñar e implementar un esquema híbrido que combine búsqueda heurística 3D (A* ponderado como guía global) con un agente PPO con enmascaramiento cinemático de acciones, asegurando trayectorias de bandejas eléctricas constructivamente viables con accesorios comerciales estándar y evaluadas frente a heurísticas puras y al trazado manual. | **Hipótesis Específica 3:**<br>La combinación de guía heurística global con una política PPO restringida mediante enmascaramiento de acciones evita óptimos locales en corredores congestionados, garantiza que el 100% de los quiebres correspondan a curvas comerciales normalizadas (30°, 45°, 60° y 90°) y supera en constructabilidad y consolidación troncal al algoritmo A* 3D convencional. | **Variables Independientes (X3):**<br>• Campo guía heurístico A* 3D<br>• Máscara de acciones cinemáticas PPO<br>• Penalización por giro/accesorio y bono de bundling<br><br>**Variable Dependiente (Y3):**<br>Viabilidad constructiva y optimalidad de la ruta 3D. | • Porcentaje de accesorios estándar utilizados (30°, 45°, 60° y 90°, meta 100%).<br>• Número de cambios de dirección por circuito.<br>• Recompensa acumulada y tasa de éxito por episodio. |

In [ ]:
# === CELDA 1: VERIFICACIÓN DE ARTEFACTOS Y RESULTADOS EXPERIMENTALES ===
from pathlib import Path
import pandas as pd

# Detectar automáticamente la raíz del repositorio TESIS-IA
repo_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("Raíz del repositorio:", repo_dir)

df_cables = pd.read_csv(repo_dir / "data" / "schedules" / "cable_schedule.csv")
df_metrics = pd.read_csv(repo_dir / "logs" / "metrics_experimentos.csv")

print(f"\nInventario cargado: {len(df_cables)} circuitos alimentadores hospitalarios.")
display(df_metrics[['exp_id', 'variante', 'num_features', 'rmse_mean', 'rmse_std', 'mae_mean', 'latencia_ms']])

In [ ]:
# === CELDA 2: SINCRONIZAR PLAN DE TESIS (.DOCX), MATRIZ DE CONSISTENCIA Y README ===
import shutil

thesis_dir = repo_dir / "docs" / "thesis"
thesis_dir.mkdir(parents=True, exist_ok=True)
downloads_dir = Path.home() / "Downloads"

# Buscar el archivo Word corregido en Descargas o en la carpeta del repo
candidatos_docx = [
    downloads_dir / "Informe_Parcial_PI2_Plan_Tesis_Juan_Villegas_v2.docx",
    repo_dir / "Informe_Parcial_PI2_Plan_Tesis_Juan_Villegas_v2.docx",
    downloads_dir / "Informe_Parcial_PI2_Plan_Tesis_Juan_Villegas.docx",
    repo_dir / "Informe_Parcial_PI2_Plan_Tesis_Juan_Villegas.docx",
]

for src in candidatos_docx:
    if src.exists():
        dst = thesis_dir / "Informe_Parcial_PI2_Plan_Tesis_Juan_Villegas.docx"
        shutil.copy2(src, dst)
        print(f"[OK] Documento Word copiado en: {dst}")
        break
else:
    print("[AVISO] No se encontró el .docx en Descargas. Verifica que esté descargado.")

# Actualizar docs/matriz_consistencia.md
matriz_md = """# Matriz de Consistencia Metodológica

**Proyecto:** Optimización del Ruteo de Bandejas Eléctricas en Entornos OpenBIM mediante Voxelización Semántica, Búsqueda Heurística y Aprendizaje por Refuerzo Profundo  
**Tesista:** Ing. Juan José Villegas Calvay  
**Asesor:** María Fernanda Tejada Begazo  
**Programa:** Maestría en Inteligencia Artificial – Universidad Nacional de Ingeniería (UNI)  

| Problema | Objetivo | Hipótesis | Variable | Indicador medible |
| :--- | :--- | :--- | :--- | :--- |
| **Problema General:**<br>¿En qué medida la integración de voxelización semántica, búsqueda heurística y aprendizaje por refuerzo profundo (PPO) sobre modelos OpenBIM (IFC) permite optimizar el trazado 3D de bandejas eléctricas de alimentadores en edificaciones hospitalarias de alta complejidad (Categoría III-1), eliminando interferencias espaciales y reduciendo los tiempos de diseño respecto al método manual y a la búsqueda heurística aislada? | **Objetivo General:**<br>Desarrollar e implementar un modelo híbrido de optimización basado en voxelización semántica, búsqueda heurística y aprendizaje por refuerzo profundo (PPO con enmascaramiento de acciones) sobre estándar OpenBIM (IFC 4 / ISO 16739) para automatizar el ruteo tridimensional de bandejas eléctricas de alimentadores en hospitales Categoría III-1, logrando cero colisiones geométricas, estricto cumplimiento normativo (CNE-Utilización, NEC Art. 392 y NEMA VE 2) y reducción del tiempo de trazado frente al diseño manual. | **Hipótesis General:**<br>La integración de un entorno espacial voxelizado semánticamente con campos ESDF, guiado por búsqueda heurística y optimizado mediante PPO con restricciones normativas explícitas, permite generar trazados 3D de bandejas eléctricas libres de colisiones (0 clashes), con 100% de cumplimiento de radio de curvatura, holgura superior NEMA VE 2 (≥ 300 mm) y llenado ≤ 40%, reduciendo en ≥ 60% el tiempo de diseño frente al trazado manual. | **Variable Independiente (X):**<br>Arquitectura híbrida de ruteo 3D (Voxelización Semántica IFC + Guía Heurística A* + Agente PPO con Action Masking).<br><br>**Variable Dependiente (Y):**<br>Desempeño técnico y constructabilidad del trazado de bandejas eléctricas. | • Conteo de colisiones duras (hard clashes = 0).<br>• Cumplimiento de holgura superior NEMA VE 2 (≥ 300 mm).<br>• Cumplimiento de radio mínimo de curvatura (R ≥ Rmín, 100%).<br>• Factor de llenado de bandeja (Rfill_ref ≤ 40%).<br>• Longitud total de bandeja (m) y tiempo de cómputo (s). |
| **Problema Específico 1:**<br>¿Cómo formular y validar un modelo de caracterización físico-normativa de los conductores y bandejas eléctricas que elimine la multicolinealidad entre parámetros dimensionales y cuantifique las variables críticas de ocupación y rigidez mecánica para alimentar la función de recompensa del agente? | **Objetivo Específico 1:**<br>Formular y validar mediante análisis de colinealidad (VIF) e ingeniería de atributos un modelo de caracterización físico-normativa de los circuitos alimentadores eléctricos, seleccionando con evidencia estadística las variables gobernantes de ocupación en bandeja (`R_fill_ref`), carga mecánica (`log_peso_total`) y margen de curvatura (`MS_curv_ref`) para parametrizar las restricciones y recompensas del entorno 3D. | **Hipótesis Específica 1:**<br>La depuración de variables colineales (VIF < 10) y la incorporación de atributos normativos derivados (`R_fill_ref`, `MS_curv_ref` y carga logarítmica) reducen el error de predicción del esfuerzo físico-constructivo respecto a las variables crudas del cuadro de cargas y concentran más del 80% de la importancia explicativa (Gain), proveyendo una señal de recompensa físicamente consistente. | **Variables Independientes (X1):**<br>• Ratio de ocupación (`R_fill_ref`)<br>• Margen de curvatura (`MS_curv_ref`)<br>• Carga lineal (`log_peso_total`)<br>• Diámetro exterior depurado por VIF<br><br>**Variable Dependiente (Y1):**<br>Índice de esfuerzo físico y ocupación normativa (`y_esfuerzo`). | • Factor de Inflación de Varianza (VIF < 10 en variables retenidas).<br>• Importancia relativa acumulada (Gain > 80% en variables creadas).<br>• Error Absoluto Medio (MAE) y RMSE en 3-Fold CV sin data leakage. |
| **Problema Específico 2:**<br>¿De qué manera la voxelización semántica y el cálculo de campos de distancia signada euclidiana (ESDF) a partir de geometrías OpenBIM (IFC) permiten representar los plenos técnicos hospitalarios saturados para detectar colisiones y holguras normativas (300 mm NEMA VE 2) de forma eficiente? | **Objetivo Específico 2:**<br>Construir un motor de ingestión y voxelización semántica 3D que extraiga elementos arquitectónicos, estructurales y MEP desde modelos IFC (`IfcOpenShell`) y calcule campos de distancia euclidiana (ESDF) para evaluar colisiones y holguras mínimas en tiempo real durante la exploración del agente. | **Hipótesis Específica 2:**<br>La discretización espacial mediante voxelización semántica multiclase (resolución ≤ 50 mm) acoplada a campos continuos ESDF permite consultar distancias a obstáculos en tiempo constante O(1) por paso, garantizando cero intersecciones geométricas y el respeto de la holgura superior libre de 300 mm (NEMA VE 2). | **Variables Independientes (X2):**<br>• Resolución de grilla voxelizada (Δv)<br>• Etiqueta semántica de entidad IFC<br><br>**Variable Dependiente (Y2):**<br>Detección de interferencias y gálibo libre en plenos técnicos. | • Tiempo de consulta espacial ESDF (ms, orden O(1)).<br>• Distancia libre vertical a losa/fondo de viga (≥ 300 mm).<br>• Conteo de intersecciones geométricas con ductos/tuberías MEP. |
| **Problema Específico 3:**<br>¿Cómo integrar una guía de búsqueda heurística con una política de aprendizaje por refuerzo profundo (PPO) dotada de enmascaramiento de acciones (action masking) para generar trayectorias de bandejas eléctricas compatibles con accesorios comerciales estándar (30°, 45°, 60° y 90°) y criterios de empaquetamiento (bundling)? | **Objetivo Específico 3:**<br>Diseñar e implementar un esquema híbrido que combine búsqueda heurística 3D (A* ponderado como guía global) con un agente PPO con enmascaramiento cinemático de acciones, asegurando trayectorias de bandejas eléctricas constructivamente viables con accesorios comerciales estándar y evaluadas frente a heurísticas puras y al trazado manual. | **Hipótesis Específica 3:**<br>La combinación de guía heurística global con una política PPO restringida mediante enmascaramiento de acciones evita óptimos locales en corredores congestionados, garantiza que el 100% de los quiebres correspondan a curvas comerciales normalizadas (30°, 45°, 60° y 90°) y supera en constructabilidad y consolidación troncal al algoritmo A* 3D convencional. | **Variables Independientes (X3):**<br>• Campo guía heurístico A* 3D<br>• Máscara de acciones cinemáticas PPO<br>• Penalización por giro/accesorio y bono de bundling<br><br>**Variable Dependiente (Y3):**<br>Viabilidad constructiva y optimalidad de la ruta 3D. | • Porcentaje de accesorios estándar utilizados (30°, 45°, 60° y 90°, meta 100%).<br>• Número de cambios de dirección por circuito.<br>• Recompensa acumulada y tasa de éxito por episodio. |
"""
(repo_dir / "docs" / "matriz_consistencia.md").write_text(matriz_md, encoding="utf-8")
print("[OK] docs/matriz_consistencia.md actualizado.")

# Actualizar README.md principal
readme_md = """# Optimización del Ruteo de Bandejas Eléctricas en Entornos OpenBIM mediante Voxelización Semántica, Búsqueda Heurística y Aprendizaje por Refuerzo Profundo

**Programa:** Maestría en Inteligencia Artificial  
**Curso:** Proyecto de Investigación II (Dr. Glen Darío Rodríguez Rafael)  
**Institución:** Universidad Nacional de Ingeniería (UNI) – Unidad de Posgrado FIIS  
**Tesista:** Ing. Juan José Villegas Calvay ([Juanvc07](https://github.com/Juanvc07))  
**Asesor:** María Fernanda Tejada Begazo  
**Entregable Consolidado:** [`docs/thesis/Informe_Parcial_PI2_Plan_Tesis_Juan_Villegas.docx`](docs/thesis/Informe_Parcial_PI2_Plan_Tesis_Juan_Villegas.docx) | [`notebooks/Plan_Tesis_y_Protocolo_PI2.ipynb`](notebooks/Plan_Tesis_y_Protocolo_PI2.ipynb)

---

## 1. Descripción del Proyecto y Planteamiento de la Problemática

En la industria de la construcción y el modelado BIM (*Building Information Modeling*), el diseño y la coordinación espacial de las **bandejas eléctricas portacables** para alimentadores principales se realiza tradicionalmente de manera manual, fragmentada y reactiva. En **establecimientos de salud de alta complejidad (Hospitales Categoría III-1)**, los plenos técnicos entre el cielo raso y las estructuras portantes presentan una saturación extrema debido a la convergencia de ductos HVAC, vapor, gases medicinales, redes contraincendios y la segregación obligatoria entre subsistemas eléctricos (Normal, Emergencia y UPS).

### Arquitectura Híbrida Propuesta (2 Fases)
1. **Fase 1 (Completada — Caracterización Físico-Normativa Tabular):** Modelado de ocupación en bandeja (`R_fill_ref`), carga mecánica (`log_peso_total`) y margen de curvatura (`MS_curv_ref`) sobre el inventario real de 24 alimentadores hospitalarios (`data/schedules/cable_schedule.csv`), con control de colinealidad (VIF) y validación cruzada sin fuga de datos (*zero data leakage*).
2. **Fase 2 (En Desarrollo — Motor Espacial OpenBIM + IA):** Extracción geométrica B-Rep desde IFC 4 mediante `IfcOpenShell`, voxelización semántica (50 mm) con Campos de Distancia Signada Euclidianos (**ESDF**), guía macro-heurística **A* 3D** y planificación secuencial con **Proximal Policy Optimization (PPO)** dotado de enmascaramiento cinemático de acciones (*Action Masking*).

---

## 2. Resultados Reales de Experimentos A/B (3-Fold Cross-Validation)

| ID Exp. | Variante de Datos (1 cambio por vez) | Features | RMSE (3-Fold CV) | MAE Medio | Latencia | Decisión |
| :--- | :--- | :---: | :---: | :---: | :---: | :---: |
| **EXP-00** | Baseline (Set A: Datos Crudos) | 5 | 18.5613 ± 12.2989 | 7.8255 | 154.73 ms | Descartado |
| **EXP-01** | Variante 1 (Set A+B: Transformados + Filtro VIF) | 6 | 16.5705 ± 15.9042 | 8.1397 | 162.31 ms | Intermedio |
| **EXP-02** | **Variante 2 (Set A+B+C: Full Dominio FE)** | 7 | **16.7858 ± 17.6488** | **7.6332** | **165.76 ms** | **ADOPTADA** |

---

## 3. Documentación del Plan de Tesis y Entregables

* **Plan de Tesis Oficial (Word):** [`docs/thesis/Informe_Parcial_PI2_Plan_Tesis_Juan_Villegas.docx`](docs/thesis/Informe_Parcial_PI2_Plan_Tesis_Juan_Villegas.docx)
* **Cuaderno Ejecutivo del Plan de Tesis:** [`notebooks/Plan_Tesis_y_Protocolo_PI2.ipynb`](notebooks/Plan_Tesis_y_Protocolo_PI2.ipynb)
* **Matriz de Consistencia Metodológica:** [`docs/matriz_consistencia.md`](docs/matriz_consistencia.md)
* **Reporte Técnico de Importancia de Variables y VIF:** [`docs/reporte_importancia_variables.md`](docs/reporte_importancia_variables.md)
* **Cuaderno de EDA y Ablación Experimental:** [`notebooks/EDA_basico.ipynb`](notebooks/EDA_basico.ipynb)
* **Estado del Arte (13 artículos y tesis base):** [`docs/papers/`](docs/papers/)
"""
(repo_dir / "README.md").write_text(readme_md, encoding="utf-8")
print("[OK] README.md actualizado.")

In [ ]:
# === CELDA 3: SUBIR TODO A GITHUB DIRECTAMENTE DESDE ESTE NOTEBOOK ===
import subprocess

def run_git(cmd):
    res = subprocess.run(cmd, cwd=repo_dir, text=True, capture_output=True)
    print(f"$ {' '.join(cmd)}")
    if res.stdout:
        print(res.stdout.strip())
    if res.stderr:
        print(res.stderr.strip())

# Crear o cambiar a la rama del Plan de Tesis (Semana 8)
run_git(["git", "checkout", "-B", "feature/semana8-plan-tesis-parcial"])

# Agregar el Word, el Notebook, la Matriz de Consistencia y el README
run_git(["git", "add", "docs/thesis/", "docs/matriz_consistencia.md", "README.md", "notebooks/Plan_Tesis_y_Protocolo_PI2.ipynb"])
run_git(["git", "status", "-s"])

# Hacer commit y subir a GitHub
run_git(["git", "commit", "-m", "docs(thesis): add partial thesis plan docx, executive notebook and updated matrix"])
run_git(["git", "push", "-u", "origin", "feature/semana8-plan-tesis-parcial"])

## 3. Referencias Bibliográficas (Repositorio `docs/papers/`)

* Biswas, S., Casas, S., Sykora, Q., Agro, B., Sadat, A., & Urtasun, R. (2024). QuAD: Query-based interpretable neural motion planning for autonomous driving. *arXiv preprint arXiv:2404.01486*. https://doi.org/10.48550/arXiv.2404.01486
* Choy, C., Gwak, J., & Savarese, S. (2019). 4D spatio-temporal ConvNets: Minkowski convolutional neural networks. *Proceedings of the IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR)*, 3075–3084.
* Codevilla, F., Müller, M., López, A., Koltun, V., & Dosovitskiy, A. (2018). End-to-end driving via conditional imitation learning. *2018 IEEE International Conference on Robotics and Automation (ICRA)*, 4693–4700. https://doi.org/10.1109/ICRA.2018.8460487
* Damaso, K. M. (2023). *Automatic routing of submarine electrical cables using machine learning* [Tesis de maestría, Naval Postgraduate School].
* Deng, R., Li, X., & Tian, Y. (2025). A review: The application of path optimization algorithms in building mechanical, electrical, and plumbing pipe design. *Buildings*, 15(12), 2093. https://doi.org/10.3390/buildings15122093
* Fadoul, A., Tizani, W., & Osorio-Sandoval, C. A. (2021). A knowledge-based model for constructability assessment of buildings design using BIM. En E. Toledo Santos & S. Scheer (Eds.), *Proceedings of the 18th International Conference on Computing in Civil and Building Engineering (ICCCBE 2020)* (pp. 147–159). Springer. https://doi.org/10.1007/978-3-030-51295-8_13
* Fait, D., Mašek, V., & Hofrichterová, P. (2024). Integrating reinforcement learning into pipe system design. *2024 8th International Symposium on Innovative Approaches in Smart Technologies (ISAS)*, 1–6. https://doi.org/10.1109/ISAS64331.2024.10845521
* Hagedorn, P., & König, M. (2021). Rule-based semantic validation for standardized linked building models. En E. Toledo Santos & S. Scheer (Eds.), *Proceedings of the 18th International Conference on Computing in Civil and Building Engineering (ICCCBE 2020)* (pp. 772–787). Springer. https://doi.org/10.1007/978-3-030-51295-8_53
* Jiang, Y. (2024). *Automatic BIM conflict resolution using a reinforcement learning approach* [Tesis de maestría, Technical University of Munich]. Chair of Computational Modeling and Simulation.
* Karlsson, T., Åblad, E., Hermansson, T., Carlson, J. S., & Tenfält, G. (2023). Automatic cable harness layout routing in a customizable 3D environment. *arXiv preprint arXiv:2311.09061*. https://doi.org/10.48550/arXiv.2311.09061
* Mirhoseini, A., Goldie, A., Yazgan, M., Jiang, J. W., Songhori, E., Wang, S., Lee, Y.-J., Johnson, E., Pathak, O., Nazi, A., Pak, J., Tong, A., Srinivasa, K., Hang, W., Tuncer, E., Le, Q. V., Laudon, J., Ho, R., Carpenter, R., & Dean, J. (2021). A graph placement methodology for fast chip design. *Nature*, 594(7862), 207–212. https://doi.org/10.1038/s41586-021-03544-w
* Rosa, A. C., Falqueiro, M. C., Bonacin, R., de Mendonça, F. L. L., Filho, G. P. R., & Gonçalves, V. P. (2023). EvacuAI: An analysis of escape routes in indoor environments with the aid of reinforcement learning. *Sensors*, 23(21), 8892. https://doi.org/10.3390/s23218892
* Singh, J., & Cheng, J. C. P. (2021). Automating the generation of 3D multiple pipe layout design using BIM and heuristic search methods. En E. Toledo Santos & S. Scheer (Eds.), *Proceedings of the 18th International Conference on Computing in Civil and Building Engineering (ICCCBE 2020)* (pp. 54–72). Springer. https://doi.org/10.1007/978-3-030-51295-8_6